In [13]:
import numpy as np
from Autograd import WhyyTorch as wt,cross_entropy_loss
import matplotlib.pyplot as plt

In [25]:
block_size = 8
words = open('bigram.txt').read().splitlines()
stoi = {'.': 0, **{chr(ord('a') + i): i + 1 for i in range(26)}} 
itos = {i: c for c, i in stoi.items()}
vocab_size = len(itos)
#Random For Our Shuffled Words, also randomizing words to avoid learning order
np.random.seed(42)
shuffled_words = words
np.random.shuffle(shuffled_words)

n1 = int(0.8 * len(shuffled_words))
n2 = int(0.9 * len(shuffled_words))

def build_split(word_list):
    Xs, Ys = [], []
    for w in word_list:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            Xs.append(context)
            Ys.append(ix)
            context = context[1:] + [ix]
    return np.array(Xs, dtype=np.int64), np.array(Ys, dtype=np.int64)

#train, dev, test split
Xtr,  Ytr  = build_split(shuffled_words[:n1])
Xdev, Ydev = build_split(shuffled_words[n1:n2])
Xte,  Yte  = build_split(shuffled_words[n2:])

print(f"train: {Xtr.shape}, dev: {Xdev.shape}, test: {Xte.shape}")

train: (182671, 8), dev: (22784, 8), test: (22691, 8)


In [15]:
Xtr[:5],Ytr[:10]

(array([[ 0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0, 10],
        [ 0,  0,  0,  0,  0,  0, 10,  1],
        [ 0,  0,  0,  0,  0, 10,  1,  8],
        [ 0,  0,  0,  0, 10,  1,  8,  2]]),
 array([10,  1,  8,  2,  1, 18,  9,  0,  4, 15]))

In [28]:
class Linear:
    def __init__(self,fan_in,fan_out,bias=True):
        self.weight = wt(np.random.randn(fan_in,fan_out) / fan_in**0.5)
        self.bias = wt(np.random.randn(fan_out)) if bias else None
        
    def __call__(self,x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out
    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias]) 
    
class BatchNorm1:
    def __init__(self,dim,eps=1e-5,momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        
        #trained for streching and shifting , to remember our normalization
        self.gamma = wt(np.ones(dim))
        self.beta = wt(np.zeros(dim))
        
        # Buffers (not part of autograd) — updated during training only
        self.running_mean = np.zeros(dim, dtype=np.float32)
        self.running_var = np.ones(dim, dtype=np.float32)
    
    def __call__(self,x):
        if self.training:
            if len(x.shape) == 3:
                xmean = x.mean((0, 1), keepdims=True)
                xvar = x.var((0, 1), keepdims=True)
            else:
                xmean = x.mean(0, keepdims=True)
                xvar = x.var(0, keepdims=True)
        else:
            if len(x.shape) == 3:
                xmean = wt(self.running_mean.reshape(1, 1, -1), requires_grad=False)
                xvar = wt(self.running_var.reshape(1, 1, -1), requires_grad=False)
            else:
                xmean = wt(self.running_mean.reshape(1, -1), requires_grad=False)
                xvar = wt(self.running_var.reshape(1, -1), requires_grad=False)
        xhat = (x - xmean) / (xvar + self.eps).sqrt()
        self.out = self.gamma * xhat + self.beta
        if self.training:
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean.data.reshape(-1)
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar.data.reshape(-1)
            
        return self.out
    
    
    def parameters(self):
        return [self.gamma,self.beta]
    
class Tanh:
    def __call__(self, x):
        self.out = x.tanh()
        return self.out
    def parameters(self):
        return []

class Embedding:
    def __init__(self,num_embeddings,embedding_dim):
        # WhyyTorch so indexing builds a graph and grads update the table
        self.weight = wt(np.random.randn(num_embeddings, embedding_dim))
    def __call__(self,IX):
        self.out = self.weight[IX]
        return self.out
    def parameters(self):
        return [self.weight]
    
class FlattenConsecutive:
    def __init__(self,n):
        self.n = n
        
    def __call__(self,x):
        self.out = x.reshape(x.shape[0],x.shape[1]//self.n,self.n*x.shape[2])
        if self.out.shape[1] == 1:
            self.out = self.out.reshape(self.out.shape[0],-1)
        return self.out
    def parameters(self):
        return []
    
class Sequential:
    def __init__(self,layers):
        self.layers = layers
    def __call__(self,x):
        for layer in self.layers:
            x = layer(x)
        self.out = x
        return self.out
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [29]:
n_embd = 24
n_hidden = 200

model = Sequential([
    Embedding(vocab_size, n_embd),

    #Stage1
    FlattenConsecutive(2),
    Linear(2*n_embd, n_hidden, bias=False),
    BatchNorm1(n_hidden),
    Tanh(),
    #Stage2
    FlattenConsecutive(2),
    Linear(2*n_hidden, n_hidden, bias=False),
    BatchNorm1(n_hidden),
    Tanh(),
    #Stage3
    FlattenConsecutive(2),
    Linear(2*n_hidden, n_hidden, bias=False),
    BatchNorm1(n_hidden),
    Tanh(),

    Linear(n_hidden, vocab_size),
])

parameters = model.parameters()
print(sum(p.data.size for p in parameters))

176875


In [31]:
# same optimization as last time
max_steps = 2000
batch_size = 32
eval_every = 50
lossi = []

def set_bn_training(model, training=True):
    for layer in model.layers:
        if isinstance(layer, BatchNorm1):
            layer.training = training

def average_loss(model, X, Y, batch_size=256):
    set_bn_training(model, False)
    losses = []
    for start in range(0, X.shape[0], batch_size):
        Xb, Yb = X[start:start + batch_size], Y[start:start + batch_size]
        losses.append(float(cross_entropy_loss(model(Xb), Yb).data))
    set_bn_training(model, True)
    return float(np.mean(losses))

for i in range(max_steps):

    # minibatch construct
    ix = np.random.randint(0, Xtr.shape[0], size=batch_size)
    Xb, Yb = Xtr[ix], Ytr[ix]   # batch X, Y

    # forward pass
    logits = model(Xb)
    loss = cross_entropy_loss(logits, Yb)       # loss function

    # backward pass
    for p in parameters:
        p.zero_grad()

    loss.backward()

    # update: simple SGD
    lr = 0.1 if i < 1000 else 0.01    # step learning rate decay

    for p in parameters:
        p.data += -lr * p.grad

    # track stats
    if i % eval_every == 0:
        tr = average_loss(model, Xtr[:5000], Ytr[:5000])
        va = average_loss(model, Xdev, Ydev)
        print(f"step {i:4d} | train {tr:.4f} | val {va:.4f}")

    lossi.append(np.log10(float(loss.data)))

step    0 | train 2.2390 | val 2.2208
step   50 | train 2.2765 | val 2.2567
step  100 | train 2.2867 | val 2.2489
step  150 | train 2.2803 | val 2.2586
step  200 | train 2.2910 | val 2.2777
step  250 | train 2.2828 | val 2.2619
step  300 | train 2.2748 | val 2.2524
step  350 | train 2.2707 | val 2.2372
step  400 | train 2.2702 | val 2.2428
step  450 | train 2.2851 | val 2.2610
step  500 | train 2.2676 | val 2.2353
step  550 | train 2.2471 | val 2.2279
step  600 | train 2.2797 | val 2.2471
step  650 | train 2.2734 | val 2.2438
step  700 | train 2.2700 | val 2.2587
step  750 | train 2.2381 | val 2.2220
step  800 | train 2.2492 | val 2.2306
step  850 | train 2.3015 | val 2.2664
step  900 | train 2.2502 | val 2.2210
step  950 | train 2.2388 | val 2.2112
step 1000 | train 2.2561 | val 2.2304
step 1050 | train 2.2064 | val 2.1762
step 1100 | train 2.1969 | val 2.1669
step 1150 | train 2.1886 | val 2.1593
step 1200 | train 2.1872 | val 2.1556
step 1250 | train 2.1867 | val 2.1518
step 1300 | 

In [ ]:
print("train:", average_loss(model, Xtr[:5000], Ytr[:5000]))
print("val:  ", average_loss(model, Xdev, Ydev))

In [46]:
# Sample names from the trained model
for layer in model.layers:
    if isinstance(layer, BatchNorm1):
        layer.training = False

for _ in range(10):
    out = []
    context = [0] * block_size

    while True:
        X = np.array([context], dtype=np.int64)
        logits = model(X)

        counts = logits.exp()
        probs = counts / counts.sum(axis=1, keepdims=True)
        ix = int(np.random.choice(vocab_size, p=probs.data.ravel()))

        context = context[1:] + [ix]
        out.append(itos[ix])
        if ix == 0:
            break

    print(''.join(out))

for layer in model.layers:
    if isinstance(layer, BatchNorm1):
        layer.training = True

brandel.
marinti.
aliana.
odarin.
dislaid.
taylanna.
methegys.
mokhu.
mayvon.
retan.
